In [ ]:
# ---------------------------------------------------------
# This code was executed in my environment to verify your file
# ---------------------------------------------------------
def chunk_by_train_schedule(text):
    # Split by the start marker
    raw_chunks = text.split('<TRAIN_START>')
    cleaned_chunks = []
    for chunk in raw_chunks:
        if "TRAIN NO:" in chunk:
            # Clean and re-assemble
            content = chunk.split('<TRAIN_END>')[0].strip()
            full_chunk = f"<TRAIN_START>\n{content}"
            cleaned_chunks.append(full_chunk)
    return cleaned_chunks

# Read your file
with open("/content/All_Trains_Cleaned.txt", "r") as f:
    full_text = f.read()

# Create chunks
train_chunks = chunk_by_train_schedule(full_text)

print(f"✅ Success! Created {len(train_chunks)} distinct train schedule chunks.")
print("--- Sample Chunk 1 ---")
print(train_chunks[1000] + "...") # Preview first 300 chars

In [ ]:
pip install google-generativeai chromadb

In [ ]:
import google.generativeai as genai

print("Listing available generative models that support 'generateContent':")
for m in genai.list_models():
    if 'generateContent' in m.supported_generation_methods:
        print(f"  - {m.name} (Supports: {m.supported_generation_methods})")

In [ ]:
!pip install torch sentence-transformers google-generativeai

In [ ]:
import os
import sys
import re
import torch
import google.generativeai as genai
from sentence_transformers import SentenceTransformer

# --- CONFIG ---
os.environ["GOOGLE_API_KEY"] = " "
if os.environ["GOOGLE_API_KEY"] == "YOUR_GEMINI_API_KEY_HERE":
    print("⚠️  ERROR: Set your API Key!")
    sys.exit()

genai.configure(api_key=os.environ["GOOGLE_API_KEY"], transport='rest')

# --- 1. SMART INDEXING (The Backbone) ---
def load_and_index_data(filename):
    print("🚀 Building Indexes...")
    with open(filename, 'r', encoding='utf-8') as f:
        text = f.read()

    chunks = []

    # Fast Lookup Tables
    id_map = {}          # "57254" -> Chunk Index
    name_map = []        # ("vijayawada passenger", Chunk Index)

    raw_splits = text.split('<TRAIN_START>')

    for split in raw_splits:
        if "TRAIN NO:" in split:
            clean_body = split.split('<TRAIN_END>')[0].strip()

            # Extract Meta info strictly for indexing
            t_num = re.search(r"TRAIN NO:\s*(\d+)", clean_body)
            t_name = re.search(r"TRAIN NAME:\s*(.*)", clean_body)

            # Create a clean chunk
            final_chunk = f"<TRAIN_START>\n{clean_body}"
            chunks.append(final_chunk)
            current_idx = len(chunks) - 1

            if t_num:
                id_map[t_num.group(1)] = current_idx
            if t_name:
                # Store lowercase for case-insensitive keyword search
                name_map.append((t_name.group(1).lower(), current_idx))

    print(f"✅ Indexed {len(chunks)} trains.")
    return chunks, id_map, name_map

# --- 2. VECTOR SETUP (The Fallback) ---
def setup_vector_db(chunks):
    print("🧠 Loading AI Model (BGE-Small)...")
    # Using BGE because it's better at retrieval than MiniLM
    model = SentenceTransformer('BAAI/bge-small-en-v1.5')

    # We only embed the headers to save time/noise
    # A trick: "Train 57254 Vijayawada Passenger" is better for embedding than the full table
    summaries = [c.split('\n')[1] + " " + c.split('\n')[2] for c in chunks]

    print("   Embedding summaries...")
    vectors = model.encode(summaries, convert_to_tensor=True, show_progress_bar=True)
    vectors = torch.nn.functional.normalize(vectors, p=2, dim=1)

    return vectors, model

# --- 3. THE "HYBRID ROUTER" SEARCH ---
def search(query, chunks, id_map, name_map, vectors, model):
    query = query.strip()

    # STRATEGY A: EXACT ID (The Sniper)
    # Check for any 5-digit number in the query
    id_match = re.search(r"\b(\d{5})\b", query)
    if id_match:
        train_id = id_match.group(1)
        if train_id in id_map:
            print(f"   [🎯 Strategy: Exact ID Match] Found {train_id}")
            return [chunks[id_map[train_id]]]

    # STRATEGY B: KEYWORD MATCH (The Filter)
    # Check if words in query match a Train Name exactly
    query_lower = query.lower()
    keyword_hits = []
    for t_name, idx in name_map:
        # Simple rule: if the query (e.g. "vijayawada") is inside the name
        if len(query_lower) > 4 and query_lower in t_name:
            keyword_hits.append(chunks[idx])
            if len(keyword_hits) >= 3: break # Limit results

    if keyword_hits:
        print(f"   [⚡ Strategy: Keyword Name Match] Found {len(keyword_hits)} trains")
        return keyword_hits

    # STRATEGY C: VECTOR SEARCH (The AI)
    # If A and B fail, use vectors
    print("   [🧠 Strategy: Semantic Vector Search]")
    q_vec = model.encode(query, convert_to_tensor=True)
    q_vec = torch.nn.functional.normalize(q_vec.unsqueeze(0), p=2, dim=1)

    scores = torch.mm(q_vec, vectors.transpose(0, 1))
    top_k = torch.topk(scores, k=3)

    results = [chunks[idx] for idx in top_k.indices[0].tolist()]
    return results

# --- 4. MAIN LOOP ---
def main():
    filename = "/content/All_Trains_Cleaned.txt"
    if not os.path.exists(filename):
        return print("File missing!")

    # 1. Index
    chunks, id_map, name_map = load_and_index_data(filename)

    # 2. Embed
    vectors, embedder = setup_vector_db(chunks)

    print("\n" + "="*40)
    print(" 🚅 ULTIMATE RAILWAY SYSTEM READY")
    print("="*40)

    while True:
        q = input("\nQuery: ")
        if q.lower() in ['exit', 'quit']: break

        # SEARCH
        results = search(q, chunks, id_map, name_map, vectors, embedder)

        if not results:
            print("❌ No trains found.")
            continue

        # GENERATE
        print("   [🤖 Gemini] Reading schedule...")
        context = "\n\n".join(results[:2]) # Feed top 2 results

        prompt = f"""You are a railway enquiry bot.
        User Query: {q}

        Using ONLY this schedule data:
        {context}

        Answer the user briefly."""

        try:
            model = genai.GenerativeModel('gemini-2.5-flash')
            print("\n>>> " + model.generate_content(prompt).text.strip())
        except Exception as e:
            print(f"API Error: {e}")

if __name__ == "__main__":
    main()

In [ ]:
import os
import sys
import re
import torch
import difflib
import google.generativeai as genai
from sentence_transformers import SentenceTransformer

# ==========================================
# 🛑 PASTE YOUR API KEY BELOW
# ==========================================
os.environ["GOOGLE_API_KEY"] = " "

if os.environ["GOOGLE_API_KEY"] == "YOUR_GEMINI_API_KEY_HERE" or not os.environ["GOOGLE_API_KEY"]:
    print("\n❌ CRITICAL ERROR: You forgot to paste your API Key!")
    sys.exit()

# Use 'rest' to avoid firewall/proxy/timeout issues
genai.configure(api_key=os.environ["GOOGLE_API_KEY"], transport='rest')

# ==========================================
# 1. SMART CHUNKING & INDEXING
# ==========================================
def load_and_index_data(filename):
    print("🚀 Reading Train Database...")
    with open(filename, 'r', encoding='utf-8') as f:
        text = f.read()

    display_chunks = []
    search_chunks = []
    id_map = {}
    name_list = [] # Just names for fuzzy matching
    name_to_idx = {} # Map name -> index

    raw_splits = text.split('<TRAIN_START>')

    for split in raw_splits:
        if "TRAIN NO:" in split:
            clean_body = split.split('<TRAIN_END>')[0].strip()

            # Extract Meta Data
            t_num = "Unknown"
            t_name = "Unknown"

            m_num = re.search(r"TRAIN NO:\s*(\d+)", clean_body)
            if m_num: t_num = m_num.group(1)

            m_name = re.search(r"TRAIN NAME:\s*(.*)", clean_body)
            if m_name: t_name = m_name.group(1).strip()

            # Extract Stations (Flattening table for search)
            station_names = []
            lines = clean_body.split('\n')
            for line in lines:
                if '|' in line and 'STATION NAME' not in line:
                    parts = line.split('|')
                    if len(parts) > 1:
                        station_names.append(parts[1].strip())
            all_stations_str = ", ".join(station_names)

            # ENRICHED TEXT (For Vector Search)
            search_text = f"Train {t_num} named {t_name}. Route stops at: {all_stations_str}"
            search_chunks.append(search_text)

            # RAW TEXT (For Reading)
            display_chunks.append(f"<TRAIN_START>\n{clean_body}")

            # Update Indexes
            idx = len(display_chunks) - 1
            if t_num != "Unknown":
                id_map[t_num] = idx
            if t_name != "Unknown":
                lower_name = t_name.lower()
                name_list.append(lower_name)
                name_to_idx[lower_name] = idx

    print(f"✅ Indexed {len(display_chunks)} trains.")
    return display_chunks, search_chunks, id_map, name_list, name_to_idx

# ==========================================
# 2. VECTOR SETUP (BGE Model)
# ==========================================
def setup_vector_db(search_chunks):
    print("🧠 Loading Local AI (BAAI/bge-small-en-v1.5)...")
    # Small, fast, state-of-the-art retrieval model
    model = SentenceTransformer('BAAI/bge-small-en-v1.5')

    print("   Embedding data (One-time setup)...")
    vectors = model.encode(search_chunks, convert_to_tensor=True, show_progress_bar=True)
    vectors = torch.nn.functional.normalize(vectors, p=2, dim=1)

    return vectors, model

# ==========================================
# 3. HYBRID SEARCH ROUTER (Fixed Logic)
# ==========================================
def search(query, display_chunks, id_map, name_list, name_to_idx, vectors, model):
    query = query.strip()
    query_lower = query.lower()

    # --- LAYER 1: EXACT ID MATCH (Highest Priority) ---
    # Regex to find any 5-digit number
    id_match = re.search(r"\b(\d{5})\b", query)
    if id_match:
        train_id = id_match.group(1)
        if train_id in id_map:
            print(f"   [🎯 Exact ID Match] Found Train {train_id}")
            return [display_chunks[id_map[train_id]]]

    # --- LAYER 2: FUZZY NAME MATCH (Typo Fixer) ---
    # Uses difflib to find names like "Passenge" -> "Passenger"
    # We check if the query *contains* a fuzzy match of a train name

    # 1. Quick substring check
    matches = []
    for name in name_list:
        if name in query_lower and len(name) > 4: # exact substring match
            matches.append(display_chunks[name_to_idx[name]])

    if matches:
        print(f"   [⚡ Substring Match] Found {len(matches)} trains")
        return matches[:3]

    # 2. Fuzzy check (slower but handles typos)
    # We look for close matches to the *entire* query or parts of it
    potential_matches = difflib.get_close_matches(query_lower, name_list, n=1, cutoff=0.7)
    if potential_matches:
        best_match = potential_matches[0]
        print(f"   [✨ Fuzzy Match] Did you mean '{best_match}'?")
        return [display_chunks[name_to_idx[best_match]]]

    # --- LAYER 3: VECTOR SEARCH (Semantic) ---
    # Handles "Trains via Mumbai", "Morning trains", etc.
    print("   [🧠 Semantic Search] Scanning routes & context...")
    q_vec = model.encode(query, convert_to_tensor=True)
    q_vec = torch.nn.functional.normalize(q_vec.unsqueeze(0), p=2, dim=1)

    scores = torch.mm(q_vec, vectors.transpose(0, 1))
    top_k = torch.topk(scores, k=3)

    results = [display_chunks[idx] for idx in top_k.indices[0].tolist()]
    return results

# ==========================================
# 4. MAIN LOOP WITH MEMORY
# ==========================================
def main():
    filename = "/content/All_Trains_Cleaned.txt"
    if not os.path.exists(filename):
        return print("File missing!")

    # Load Data
    display_chunks, search_chunks, id_map, name_list, name_to_idx = load_and_index_data(filename)
    vectors, embedder = setup_vector_db(search_chunks)

    # Chat History Buffer
    chat_history = []

    print("\n" + "="*50)
    print(" 🚅 AI RAILWAY ASSISTANT (With Memory & Typos)")
    print("="*50)

    while True:
        try:
            q = input("\nQuery: ")
            if q.lower() in ['exit', 'quit']: break
            if not q: continue

            # SEARCH
            results = search(q, display_chunks, id_map, name_list, name_to_idx, vectors, embedder)

            if not results:
                print("❌ No trains found.")
                continue

            # PREPARE PROMPT
            print("   [🤖 Gemini] Thinking...")

            # Combine retrieved chunks
            context = "\n\n".join(results[:2])

            # Format History for the LLM
            history_text = "\n".join([f"User: {h[0]}\nAI: {h[1]}" for h in chat_history[-3:]])

            prompt = f"""You are a smart railway assistant.

            --- CONVERSATION HISTORY ---
            {history_text}

            --- NEW RETRIEVED DATA ---
            {context}

            --- USER QUERY ---
            {q}

            INSTRUCTIONS:
            1. Answer the User Query based on the RETRIEVED DATA.
            2. If the user refers to the previous conversation (e.g., "what is its number?"), use the CONVERSATION HISTORY.
            3. Be concise.
            """

            # GENERATE (Using Stable 1.5 Flash)
            model = genai.GenerativeModel('gemini-2.5-flash')
            response = model.generate_content(prompt)
            answer = response.text.strip()

            print("\n>>> " + answer)

            # Update History
            chat_history.append((q, answer))

        except Exception as e:
            print(f"\n⚠️ Error: {e}")

if __name__ == "__main__":
    main()

In [ ]:
 🚀 Reading Train Database...
✅ Indexed 5191 trains.
🧠 Loading Local AI (BAAI/bge-small-en-v1.5)...
   Embedding data (One-time setup)...
Batches: 100% 163/163 [09:45<00:00,  1.54it/s]
==================================================
 🚅 AI RAILWAY ASSISTANT (With Memory & Typos)
==================================================

Query: trains from Mumbai to delhi
   [🧠 Semantic Search] Scanning routes & context...
   [🤖 Gemini] Thinking...

>>> The Mumbai Central-New Delhi Rajdhani Express (Train No: 12951) goes from Mumbai to Delhi. It departs from Mumbai Central at 16:40:00 on Day 1 and arrives at New Delhi at 08:30:00 on Day 2.

Query: anyother train?
   [🧠 Semantic Search] Scanning routes & context...
   [🤖 Gemini] Thinking...

>>> I don't have information about other trains from Mumbai to Delhi in the retrieved data.

Query: 511196 train details
   [🧠 Semantic Search] Scanning routes & context...
   [🤖 Gemini] Thinking...

>>> I don't have information about train 511196 in the retrieved data.

Query: 51196 train details
   [🎯 Exact ID Match] Found Train 51196
   [🤖 Gemini] Thinking...

>>> Train 51196, the Ballarshah Wardha Mumbai Passenger, departs from BALHARSHAH at 17:30:00 on Day 1 and arrives at MUMBAI CST at 12:00:00 on Day 2. It has stops at stations including Chandrapur, Wardha Jn, Akola Jn, Bhusaval Jn, Manmad Jn, Nasik Road, Kalyan Jn, Thane, and Mumbai Dadar Central.

Query: information about 56323
   [🎯 Exact ID Match] Found Train 56323
   [🤖 Gemini] Thinking...

>>> Train 56323 is the Coimbatore Mangalore Fast Passenger. It departs from Coimbatore Jn at 07:40:00 on Day 1 and arrives at Mangalore Cntl at 20:00:00 on Day 1. It stops at stations including Palakkad, Shoranur Jn, Tirur, Kozhikode, Kannur, Payyanur, and Kasaragod.

Query: at 8:03  which trains can i pick up in route to Mumbai
   [🧠 Semantic Search] Scanning routes & context...
   [🤖 Gemini] Thinking...

>>> Based on the retrieved data, there are no trains that you can pick up en route to Mumbai. Both trains listed originate in Mumbai.

Query: which train will stop at kamkhaya  and when
   [🧠 Semantic Search] Scanning routes & context...
   [🤖 Gemini] Thinking...

>>> Both Train 15927 (Kamakhya Dibrugarh Express) and Train 15668 (Kamakhya - Gandhidham Exp) stop at Kamakhya.
Train 15927 departs at 06:50:00 on Day 1.
Train 15668 departs at 11:00:00 on Day 1.

Query: train 15646 also stops at  kamkhya why didnt you mentioned
   [🎯 Exact ID Match] Found Train 15646
   [🤖 Gemini] Thinking...

>>> Train 15646 (Guwahati - Mumbai LTT Exp) does indeed stop at Kamakhya, arriving at 17:01:00 and departing at 17:02:00 on Day 1. My apologies for not mentioning it earlier, as this information was not available in my previous search.

Query:

In [ ]:
 🚀 Reading Train Database...
✅ Indexed 5191 trains.
🧠 Loading Local AI (BAAI/bge-small-en-v1.5)...
   Embedding data (One-time setup)...
Batches: 100% 163/163 [09:45<00:00,  1.54it/s]
==================================================
 🚅 AI RAILWAY ASSISTANT (With Memory & Typos)
==================================================

Query: trains from Mumbai to delhi
   [🧠 Semantic Search] Scanning routes & context...
   [🤖 Gemini] Thinking...

>>> The Mumbai Central-New Delhi Rajdhani Express (Train No: 12951) goes from Mumbai to Delhi. It departs from Mumbai Central at 16:40:00 on Day 1 and arrives at New Delhi at 08:30:00 on Day 2.

Query: anyother train?
   [🧠 Semantic Search] Scanning routes & context...
   [🤖 Gemini] Thinking...

>>> I don't have information about other trains from Mumbai to Delhi in the retrieved data.

Query: 511196 train details
   [🧠 Semantic Search] Scanning routes & context...
   [🤖 Gemini] Thinking...

>>> I don't have information about train 511196 in the retrieved data.

Query: 51196 train details
   [🎯 Exact ID Match] Found Train 51196
   [🤖 Gemini] Thinking...

>>> Train 51196, the Ballarshah Wardha Mumbai Passenger, departs from BALHARSHAH at 17:30:00 on Day 1 and arrives at MUMBAI CST at 12:00:00 on Day 2. It has stops at stations including Chandrapur, Wardha Jn, Akola Jn, Bhusaval Jn, Manmad Jn, Nasik Road, Kalyan Jn, Thane, and Mumbai Dadar Central.

Query: information about 56323
   [🎯 Exact ID Match] Found Train 56323
   [🤖 Gemini] Thinking...

>>> Train 56323 is the Coimbatore Mangalore Fast Passenger. It departs from Coimbatore Jn at 07:40:00 on Day 1 and arrives at Mangalore Cntl at 20:00:00 on Day 1. It stops at stations including Palakkad, Shoranur Jn, Tirur, Kozhikode, Kannur, Payyanur, and Kasaragod.

Query: at 8:03  which trains can i pick up in route to Mumbai
   [🧠 Semantic Search] Scanning routes & context...
   [🤖 Gemini] Thinking...

>>> Based on the retrieved data, there are no trains that you can pick up en route to Mumbai. Both trains listed originate in Mumbai.

Query: which train will stop at kamkhaya  and when
   [🧠 Semantic Search] Scanning routes & context...
   [🤖 Gemini] Thinking...

>>> Both Train 15927 (Kamakhya Dibrugarh Express) and Train 15668 (Kamakhya - Gandhidham Exp) stop at Kamakhya.
Train 15927 departs at 06:50:00 on Day 1.
Train 15668 departs at 11:00:00 on Day 1.

Query: train 15646 also stops at  kamkhya why didnt you mentioned
   [🎯 Exact ID Match] Found Train 15646
   [🤖 Gemini] Thinking...

>>> Train 15646 (Guwahati - Mumbai LTT Exp) does indeed stop at Kamakhya, arriving at 17:01:00 and departing at 17:02:00 on Day 1. My apologies for not mentioning it earlier, as this information was not available in my previous search.

Query: list all trains from jansi
   [🧠 Semantic Search] Scanning routes & context...
   [🤖 Gemini] Thinking...

>>> I'm sorry, but based on the information I have, there are no trains listed that depart from Jansi.

Query: list all trains from  ranchi
   [🧠 Semantic Search] Scanning routes & context...
   [🤖 Gemini] Thinking...

>>> Here are the trains departing from Ranchi:
*   Train 58655 (Ranchi Lohardaga Passenger) departs at 14:45:00 on Day 1.
*   Train 58653 (Ranchi - Lohardaga - Barkichampi Passenger) departs at 09:30:00 on Day 1.

Query: tarin12453 is also from ranchi
   [🧠 Semantic Search] Scanning routes & context...
   [🤖 Gemini] Thinking...

>>> I'm sorry, but I don't have any information about Train 12453. Based on my current data, the trains departing from Ranchi are 12020 (Ranchi-Howrah Shatabdi Express) and 58655 (Ranchi Lohardaga Passenger).

Query: there is no train no as 12020
   [🎯 Exact ID Match] Found Train 12020
   [🤖 Gemini] Thinking...

>>> I apologize for the confusion. According to my updated information, Train 12020 (Ranchi-Howrah Shatabdi Express) does exist and departs from Ranchi at 13:45:00 on Day 1.

Query: so i told you to list all the trains with name departs from ranchi alll
   [🧠 Semantic Search] Scanning routes & context...
   [🤖 Gemini] Thinking...

>>> Here are the trains departing from Ranchi:
*   Train 12020 (Ranchi-Howrah Shatabdi Express) departs at 13:45:00 on Day 1.
*   Train 58655 (Ranchi Lohardaga Passenger) departs at 14:45:00 on Day 1.
*   Train 58657 (Ranchi Lohardaga Passenger) departs at 18:45:00 on Day 1.

Query: which train stops at ranchi
   [🧠 Semantic Search] Scanning routes & context...
   [🤖 Gemini] Thinking...

>>> Here are the trains that stop at Ranchi:
*   Train 12020 (Ranchi-Howrah Shatabdi Express)
*   Train 58655 (Ranchi Lohardaga Passenger)
*   Train 58657 (Ranchi Lohardaga Passenger)
*   Train 18628 (Ranchi-Howrah InterCity Express)
*   Train 58653 (Ranchi - Lohardaga - Barkichampi Passenger)

Query: this all also departures from ranchi i want you to tell me trains which have ranchi a middle stop and schedule
   [🧠 Semantic Search] Scanning routes & context...
   [🤖 Gemini] Thinking...

>>> Based on the new retrieved data, both Train 58653 (Ranchi - Lohardaga - Barkichampi Passenger) and Train 58657 (Ranchi Lohardaga Passenger) depart from Ranchi and do not have Ranchi as a middle stop. There are no trains in the provided data that have Ranchi as a middle stop.

Query: you still missed many trains from ranchi check again give me all list
   [🧠 Semantic Search] Scanning routes & context...
   [🤖 Gemini] Thinking...

>>> Here are all the trains confirmed to depart from Ranchi, along with their schedules:

*   **Train 58651** (Ranchi Lohardaga Passenger) departs at 05:30:00 on Day 1.
*   **Train 12020** (Ranchi-Howrah Shatabdi Express) departs at 13:45:00 on Day 1.
*   **Train 58655** (Ranchi Lohardaga Passenger) departs at 14:45:00 on Day 1.
*   **Train 58657** (Ranchi Lohardaga Passenger) departs at 18:45:00 on Day 1.
*   **Train 58653** (Ranchi - Lohardaga - Barkichampi Passenger) also departs from Ranchi.

Query: what about TRAIN NO: 18613
   [🎯 Exact ID Match] Found Train 18613
   [🤖 Gemini] Thinking...

>>> Train 18613 (Ranchi Chopan Express) departs from Ranchi at 07:50:00 on Day 1.

Its schedule is:
*   **RANCHI (RNC)**: Departs 07:50:00 (Day 1)
*   **NAMKON (NKM)**: Arrives 07:56:00, Departs 07:58:00 (Day 1)
*   **SILLI (SLF)**: Arrives 08:45:00, Departs 08:47:00 (Day 1)
*   **MURI (MURI)**: Arrives 09:13:00, Departs 09:18:00 (Day 1)
*   **RAMGARH CANT (RMT)**: Arrives 10:13:00, Departs 10:15:00 (Day 1)
*   **BARKAKANA (BRKA)**: Arrives 10:30:00, Departs 10:40:00 (Day 1)
*   **PATRATU (PTRU)**: Arrives 10:59:00, Departs 11:00:00 (Day 1)
*   **MCCLUSKIEGANJ (MGME)**: Arrives 11:44:00, Departs 11:45:00 (Day 1)
*   **TORI (TORI)**: Arrives 12:04:00, Departs 12:05:00 (Day 1)
*   **BARWADIH JN (BRWD)**: Arrives 13:41:00, Departs 13:46:00 (Day 1)
*   **DALTONGANJ (DTO)**: Arrives 14:13:00, Departs 14:15:00 (Day 1)
*   **GARWA ROAD (GHD)**: Arrives 14:50:00, Departs 14:55:00 (Day 1)
*   **GARHWA (GHQ)**: Arrives 15:04:00, Departs 15:05:00 (Day 1)
*   **RAMNA (RMF)**: Arrives 15:23:00, Departs 15:24:00 (Day 1)
*   **NAGAR UNTARI (NUQ)**: Arrives 15:35:00, Departs 15:36:00 (Day 1)
*   **RENUKUT (RNQ)**: Arrives 16:24:00, Departs 16:25:00 (Day 1)
*   **CHOPAN (CPU)**: Arrives 17:25:00 (Day 1)

Query: mumbai to banglore trains
   [🧠 Semantic Search] Scanning routes & context...
   [🤖 Gemini] Thinking...

>>> I am sorry, but based on the provided data, I do not have information about trains from Mumbai to Bangalore. The available data includes trains like the Deccan Express (Mumbai to Pune) and the Rupashi Bangla Express (Purulia to Howrah).

Query: trains which has banglore as middle station
   [🧠 Semantic Search] Scanning routes & context...
   [🤖 Gemini] Thinking...

>>> Based on the provided data, there are no trains listed that have "Bangalore" as a middle station.

Query: trains from banglore
   [🧠 Semantic Search] Scanning routes & context...
   [🤖 Gemini] Thinking...

>>> I am sorry, but based on the provided data, there are no trains listed that start from Bangalore.

Query:  trains from  BANGALORE
   [🧠 Semantic Search] Scanning routes & context...
   [🤖 Gemini] Thinking...

>>> Yes, I can help with that now. Based on the new data, here are two trains that start from Bangalore:

*   **Bangalore Tumkur Passenger (Train No: 56225)**, departing from Bangalore City Jn (SBC) at 13:40:00.
*   **Bangalore Mysore (PUSH-PULL) Passenger (Train No: 56238)**, departing from Bangalore City Jn (SBC) at 19:00:00.

Query: all the trains
   [🧠 Semantic Search] Scanning routes & context...
   [🤖 Gemini] Thinking...
WARNING:tornado.access:429 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 1218.72ms

⚠️ Error: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/usage?tab=rate-limit.
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash
Please retry in 1.190541395s.

Query: